# Construindo a Arquitetura da CNN 

![Extracao de características](extracao_caracteristicas.png)

## 1. Importando as bibliotecas

In [ ]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Convolution2D
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense
from keras.layers import Dropout
from keras import utils
import numpy as np

## 2. Aquisição dos dados

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

## 3. Pré-processamento

In [ ]:
X_train = X_train / 255.
X_test = X_test / 255.
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)
y_train = utils.to_categorical(y_train) #8 -> 0 0 0 0 0 0 0 0 1 0
y_test = utils.to_categorical(y_test) #3 -> 0 0 0 1 0 0 0 0 0 0

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

## 4. Arquitetura da CNN

![Arquitetura CNN](cnn_arquitetura_basica.png)

In [ ]:
# Inicializando a CNN
classifier = Sequential()

#Camada de convolução
classifier.add(Convolution2D(32, kernel_size=(3,3), input_shape = (28, 28,1), activation = 'relu', padding='same', name = 'conv_1'))

#Camada de pooling
classifier.add(MaxPooling2D(pool_size=(2,2), strides=(2, 2), padding='same', name = 'pool_1'))

#Segunda camada convolucional
classifier.add(Convolution2D(64, kernel_size=(3,3), activation = 'relu', padding='same', name = 'conv_2'))


#Segunda camada de pooling
classifier.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same', name = 'pool_2'))


#Vetorizando os mapas de características do último pooling (camada de entrada)
classifier.add(Flatten())

#Dropout
classifier.add(Dropout(0.5))

#Camada totalmente conectada ou oculta
classifier.add(Dense(activation='relu', units=128, name = 'dense_1'))


#Camada de saída
classifier.add(Dense(activation='softmax', units=10,  name = 'classification'))

In [ ]:
classifier.summary()

## 5. Treinando o modelo

In [ ]:
#Parâmetros de treinamento
epochs = 5
batch_size = 50
validation_split=0.1

In [ ]:
print(54000/50)

In [ ]:
classifier.compile(optimizer = 'sgd', loss= 'categorical_crossentropy', metrics=['accuracy'])

checkpoint = keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='auto', save_freq='epoch') 
earlystop = keras.callbacks.EarlyStopping(patience=15)

In [ ]:
classifier.fit(X_train, y_train, validation_split=validation_split, batch_size=batch_size, epochs=epochs, callbacks=[checkpoint,earlystop], verbose=1)

## 6. Avaliando o modelo

In [ ]:
best_model = keras.models.load_model("best_model.h5")

In [ ]:
score = best_model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

---
## Atividade — Treinar e Avaliar a CNN Adaptada para Fashion-MNIST

### Dataset Escolhido: Fashion-MNIST

O **Fashion-MNIST** (Zalando Research, 2017) contém 70.000 imagens em escala de cinza (28×28 px) divididas em 10 categorias de moda.

**Por que Fashion-MNIST?**
- Mesma estrutura do MNIST: 28×28 pixels, escala de cinza → `input_shape` não precisa mudar
- Mesmo número de classes (10) → camada de saída permanece igual
- Padrões visuais mais complexos → avalia melhor a capacidade da CNN

**Classes:**
```
0: Camiseta   1: Calça    2: Pullover  3: Vestido  4: Casaco
5: Sandália   6: Camisa   7: Tênis     8: Bolsa    9: Bota
```

**Adaptações feitas em relação à arquitetura original (MNIST):**

| Parâmetro | MNIST original | Fashion-MNIST | Motivo |
|---|---|---|---|
| Filtros Conv1 | 32 | 64 | Padrões mais complexos exigem mais mapas de características |
| Filtros Conv2 | 64 | 128 | Segunda camada captura detalhes mais finos |
| Unidades Dense | 128 | 256 | Maior capacidade de representação |
| Otimizador | SGD | Adam | Convergência mais rápida, essencial sem GPU |
| Épocas | 5 | 15 | Fashion-MNIST precisa de mais iterações |
| Batch size | 50 | 64 | Melhor eficiência em CPU |
| EarlyStopping patience | 15 | 5 | Evita tempo de treino excessivo em CPU |

### A.1 Carregando o Fashion-MNIST

In [ ]:
# Nomes das classes para referência
class_names = ['Camiseta', 'Calça', 'Pullover', 'Vestido', 'Casaco',
                'Sandália', 'Camisa', 'Tênis', 'Bolsa', 'Bota']

# Carregando o dataset
(X_train_f, y_train_f), (X_test_f, y_test_f) = keras.datasets.fashion_mnist.load_data()

print("Shapes brutos:")
print(X_train_f.shape)
print(y_train_f.shape)
print(X_test_f.shape)
print(y_test_f.shape)

### A.2 Pré-processamento

In [ ]:
# Normalização (0-255 -> 0-1)
X_train_f = X_train_f / 255.
X_test_f  = X_test_f  / 255.

# Adicionando canal de cor (escala de cinza = 1 canal)
X_train_f = X_train_f.reshape(X_train_f.shape[0], 28, 28, 1)
X_test_f  = X_test_f.reshape(X_test_f.shape[0],  28, 28, 1)

# One-hot encoding dos rótulos
y_train_f = utils.to_categorical(y_train_f)  # ex: 3 -> [0,0,0,1,0,0,0,0,0,0]
y_test_f  = utils.to_categorical(y_test_f)

print("Shapes após pré-processamento:")
print(X_train_f.shape)
print(y_train_f.shape)
print(X_test_f.shape)
print(y_test_f.shape)

### A.3 Arquitetura CNN Adaptada

In [ ]:
# Inicializando a CNN para Fashion-MNIST
classifier_fashion = Sequential()

# Primeira camada de convolução (64 filtros — dobro do original)
classifier_fashion.add(Convolution2D(64, kernel_size=(3,3), input_shape=(28, 28, 1),
                                     activation='relu', padding='same', name='conv_1'))

# Primeira camada de pooling
classifier_fashion.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same', name='pool_1'))

# Segunda camada de convolução (128 filtros — dobro do original)
classifier_fashion.add(Convolution2D(128, kernel_size=(3,3),
                                      activation='relu', padding='same', name='conv_2'))

# Segunda camada de pooling
classifier_fashion.add(MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same', name='pool_2'))

# Vetorizando os mapas de características
classifier_fashion.add(Flatten())

# Dropout para regularização
classifier_fashion.add(Dropout(0.5))

# Camada totalmente conectada (256 unidades — dobro do original)
classifier_fashion.add(Dense(activation='relu', units=256, name='dense_1'))

# Camada de saída (10 classes — igual ao original)
classifier_fashion.add(Dense(activation='softmax', units=10, name='classification'))

classifier_fashion.summary()

### A.4 Treinando o modelo Fashion-MNIST

In [ ]:
# Parâmetros de treinamento adaptados para CPU e Fashion-MNIST
epochs_f       = 15   # mais épocas para convergir (EarlyStopping interrompe se necessário)
batch_size_f   = 64   # batch levemente maior para eficiência em CPU
val_split_f    = 0.1  # 10% para validação

In [ ]:
# Compilando com Adam (melhor que SGD para datasets mais complexos)
classifier_fashion.compile(optimizer='adam',
                            loss='categorical_crossentropy',
                            metrics=['accuracy'])

# Callbacks: salva o melhor modelo e para cedo se não houver melhora
checkpoint_f = keras.callbacks.ModelCheckpoint(
    'best_model_fashion.h5',
    monitor='val_loss', verbose=1,
    save_best_only=True, mode='auto', save_freq='epoch'
)
earlystop_f = keras.callbacks.EarlyStopping(patience=5)  # patience menor para economizar tempo em CPU

In [ ]:
history_fashion = classifier_fashion.fit(
    X_train_f, y_train_f,
    validation_split=val_split_f,
    batch_size=batch_size_f,
    epochs=epochs_f,
    callbacks=[checkpoint_f, earlystop_f],
    verbose=1
)

### A.5 Avaliando o modelo Fashion-MNIST

In [ ]:
best_model_fashion = keras.models.load_model('best_model_fashion.h5')

score_f = best_model_fashion.evaluate(X_test_f, y_test_f, verbose=0)
print("Fashion-MNIST — Test loss:    ", round(score_f[0], 4))
print("Fashion-MNIST — Test accuracy:", round(score_f[1], 4))

### A.6 Comparação de Resultados

| Métrica | MNIST (original) | Fashion-MNIST (adaptado) |
|---|---|---|
| Test Loss | 0.0415 | *(preencher após rodar)* |
| Test Accuracy | 98.68% | *(preencher após rodar)* |

**Análise esperada:**  
O Fashion-MNIST tende a apresentar acurácia menor (~88–92%) do que o MNIST (~98%), pois suas imagens contêm padrões visuais mais sutis e variações intra-classe (ex: diferentes modelos de camiseta). Isso demonstra que, mesmo com a arquitetura ampliada, o problema é genuinamente mais difícil.